# Retrosynthesis exercise

Using the USPTO-50k dataset, here this exercise addresses the core principle of template-based retrosynthesis:

Product → Reaction template → Reactants

The first part of this notebook deals with only predicting the first step here, the reaction templates, i.e. here simply the reaction class. This simple classification model would become a policy in a multi-step search algorithm.

To make this a bit less abstract, in the second part of the notebook, reactants are recovered from the predicted reaction classes. Note that this is a simple data-driven approach and does include 
- atom mapping
- chemistry rules (hard‑coded)
- graph rewriting.

Hence, it is a conceptual demonstration, far from the prowess of more complex models. However, early systems would have adopted a very similar approach, so there is some real-world relevance of this exercise.

### Part 1: Predicting reaction classes

1) Setup

In [49]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm


2) Load Dataset

In [50]:
df = pd.read_csv("uspto-50k.csv")
df = df[["prod_smiles", "rxn_smiles", "rxn_class"]]
df["reactants"] = df["rxn_smiles"].str.split(">").str[0]

print("Number of reactions:", len(df))
df.head()

Number of reactions: 49015


,prod_smiles,rxn_smiles,rxn_class,reactants
0,COC(=O)C(C)c1ccc(-c2ccc(-c3nc(C(N)=O)c(C)nc3C)...,CC1(C)OB([c:1]2[cH:2][cH:3][c:4]([CH:5]([C:6](...,3,CC1(C)OB([c:1]2[cH:2][cH:3][c:4]([CH:5]([C:6](...
1,Cc1cc(-c2ccc(F)cc2)c2c(c1)cc1n2CCNC1=O,Br[c:1]1[cH:2][c:3]([CH3:4])[cH:5][c:6]2[c:7]1...,3,Br[c:1]1[cH:2][c:3]([CH3:4])[cH:5][c:6]2[c:7]1...
2,C[C@]1(F)CC(F)(F)[C@@](C)(c2cc(N)ccc2F)NC1=S,CC(C)(C)OC(=O)[NH:1][c:2]1[cH:3][cH:4][c:5]([F...,6,CC(C)(C)OC(=O)[NH:1][c:2]1[cH:3][cH:4][c:5]([F...
3,CCOC(=O)Cc1cccc(Oc2ccc(-c3c(C)noc3C)cc2CN2C(=O...,Br[c:1]1[c:2]([CH3:3])[n:4][o:5][c:6]1[CH3:7]....,3,Br[c:1]1[c:2]([CH3:3])[n:4][o:5][c:6]1[CH3:7]....
4,Nc1cc(CN2CCOCC2)cc(C(F)(F)F)c1,O=[C:1]([c:2]1[cH:3][c:4]([NH2:5])[cH:6][c:7](...,7,O=[C:1]([c:2]1[cH:3][c:4]([NH2:5])[cH:6][c:7](...


3) Featurisation
In retrosynthesis, only the product is known at the prediction time, so the product needs to be encoded (here: MorganFP).

In [51]:
# helper function 
def smiles_to_fp(smiles, n_bits=2048, radius=2):
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    fp = fpgen.GetFingerprint(mol)
    fp_arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    return fp_arr

In [52]:
# Build the dataset
X = []
y = []
row_indices = [] 

for idx, row in df.iterrows():
    fp = smiles_to_fp(row["prod_smiles"])
    if fp is None:
        continue
    X.append(fp)
    y.append(row["rxn_class"])
    row_indices.append(idx)

X = np.array(X)
y = np.array(y)
row_indices = np.array(row_indices)

print("Usable samples:", len(X))
print("Number of classes:", len(set(y)))

Usable samples: 49015
Number of classes: 10


In [53]:
# Optional subsampling if too high load:
MAX_SAMPLES = 10000  # 30k took about 10 mins of training on my PC, 10k about 2.5 mins

if MAX_SAMPLES:
    idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
    X = X[idx]
    y = y[idx]
    row_indices = row_indices[idx]


X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, row_indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 8000
Test size: 2000


4) Train retrosynthesis classifier
Multinomial logistic regression is fast, interpretable, probabilistic (top-k!). Other models could be used as well, e.g. RF.

In [54]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    solver="saga"
)

model.fit(X_train, y_train)

c:\Users\jschoer\Desktop\DSA104development\DSA104\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

5) Evaluate top-k accuracy 

In [55]:
def top_k_accuracy(y_true, probs, k):
    top_k = np.argsort(probs, axis=1)[:, -k:]
    return np.mean([
        y_true[i] in top_k[i]
        for i in range(len(y_true))
    ])

In [56]:
probs = model.predict_proba(X_test)

for k in [1, 3, 5]:
    acc = top_k_accuracy(y_test, probs, k)
    print(f"Top-{k} accuracy: {acc:.3f}")

Top-1 accuracy: 0.055
Top-3 accuracy: 0.290
Top-5 accuracy: 0.538


6) Inspect some results:

In [66]:
i = np.random.randint(len(X_test))

true_class = y_test[i]
predicted_classes = np.argsort(probs[i])[-5:][::-1]

print("True reaction class:", true_class)
print("Top-5 predicted classes:", predicted_classes.tolist())

True reaction class: 2
Top-5 predicted classes: [1, 2, 6, 0, 5]


7) Discussion

- Why is retrosynthesis naturally a *ranking* problem?
- Why is accuracy higher than forward prediction?
- What chemical information is missing?
- How would this integrate into a multi-step planner?

### Part 2: Approximating the reactants

Idea: For each reaction class...
- collect all reactant SMILES
- keep the most frequent ones
- later, retrieve from this list

Result: Fast and robust approximation

1) Build reactant libraries per class

In [58]:
from collections import defaultdict, Counter

class_to_reactants_freq = defaultdict(list)

for cls, idx in zip(y_train, idx_train):
    reactants = df.loc[idx, "reactants"]
    class_to_reactants_freq[int(cls)].append(reactants)

for cls in class_to_reactants_freq:
    counts = Counter(class_to_reactants_freq[cls])
    class_to_reactants_freq[cls] = counts.most_common()


Inspect reaction classes:

In [59]:
example_class = list(class_to_reactants_freq.keys())[0]

print("Reaction class:", example_class)
print("Most common reactants:")
for r, count in class_to_reactants_freq[example_class][:5]:
    print(f"- {r} ({count}×)")


Reaction class: 6
Most common reactants:
- c1ccc(C[O:1][c:2]2[c:3]([CH2:4][NH:5][C:6](=[O:7])[NH:8][CH2:9][CH2:10][OH:11])[cH:12][cH:13][cH:14][cH:15]2)cc1 (2×)
- C[O:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]([O:8][CH:9]([C:10](=[O:11])[NH:12][S:13](=[O:14])(=[O:15])[c:16]2[cH:17][cH:18][c:19]([CH:20]([CH3:21])[CH3:22])[cH:23][cH:24]2)[c:25]2[cH:26][cH:27][c:28]3[c:29]([cH:30]2)[O:31][CH2:32][O:33]3)[c:34]([CH2:35][CH:36]([CH3:37])[CH3:38])[cH:39]1 (2×)
- C[O:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]([C:8]2=[N:9][S:10][C:11]([CH3:12])([c:13]3[cH:14][c:15]([Cl:16])[cH:17][c:18]([Cl:19])[cH:20]3)[CH2:21]2)[cH:22][c:23]1[CH3:24] (1×)
- CC(C)(C)OC(=O)[N:1]1[CH2:2][CH2:3][CH:4]([C:5]([CH3:6])([CH3:7])[S:8](=[O:9])(=[O:10])[c:11]2[cH:12][cH:13][cH:14][c:15]([C:16]([F:17])([F:18])[F:19])[n:20]2)[CH2:21][CH2:22]1 (1×)
- C[O:1][C:2]([C:3]1([c:4]2[cH:5][cH:6][c:7]([NH:8][c:9]3[n:10][c:11](-[c:12]4[cH:13][cH:14][cH:15][cH:16][cH:17]4)[cH:18][c:19]([NH:20][CH3:21])[n:22]3)[cH:23][cH:24]2)[CH2:25][CH2:26

2) Predict Reactants Given a Predicted Reaction Class

In [60]:
"""
    Return top-k most frequent reactant sets
    for a given reaction class.
    """

def predict_reactants(predicted_class, k=5):
    predicted_class = int(predicted_class)
    
    if predicted_class not in class_to_reactants_freq:
        return []  # or raise a warning
    
    return [
        r for r, _ in class_to_reactants_freq[predicted_class][:k]
    ]


3) Combine with the class prediction:

In [61]:
i = np.random.randint(len(X_test))

# True values
true_class = y_test[i]

# CORRECT: map index -> class label
predicted_class = model.classes_[np.argmax(probs[i])]

predicted_reactants = predict_reactants(predicted_class, k=5)

print("True reaction class:", true_class)
print("Predicted reaction class:", predicted_class)

print("\nTop-5 predicted reactant sets:")
for r in predicted_reactants:
    print("-", r)

True reaction class: 3
Predicted reaction class: 1

Top-5 predicted reactant sets:
- Cl[c:1]1[c:2]([N+:3](=[O:4])[O-:5])[c:6]([Cl:7])[cH:8][cH:9][cH:10]1.[CH3:11][O:12][c:13]1[cH:14][cH:15][c:16]([NH2:17])[cH:18][cH:19]1
- I[c:1]1[cH:2][c:3]2[c:4]([Cl:5])[cH:6][c:7]([CH3:8])[n:9][c:10]2[cH:11][cH:12]1.[OH:13][CH2:14][c:15]1[cH:16][cH:17][cH:18][n:19][cH:20]1
- O=[C:1]1[C@@:2]2([CH3:3])[CH2:4][C@H:5]([O:6][CH3:7])[C@H:8]3[C@H:9]([C@@H:10]2[CH2:11][CH2:12]1)[CH2:13][CH2:14][c:15]1[cH:16][c:17]([OH:18])[cH:19][cH:20][c:21]13.[CH3:22][c:23]1[cH:24][cH:25][c:26]([S:27](=[O:28])(=[O:29])[NH:30][NH2:31])[cH:32][cH:33]1
- F[c:1]1[c:2]([N+:3](=[O:4])[O-:5])[cH:6][cH:7][c:8]([F:9])[cH:10]1.[NH2:11][c:12]1[cH:13][cH:14][c:15]([CH2:16][CH2:17][OH:18])[cH:19][cH:20]1
- F[c:1]1[cH:2][cH:3][c:4]([N+:5](=[O:6])[O-:7])[cH:8][cH:9]1.[CH3:10][N:11]([CH3:12])[CH2:13][CH2:14][CH:15]([OH:16])[c:17]1[cH:18][cH:19][cH:20][cH:21][cH:22]1


4) Evaluation: Were the true reactants recovered? 

Use top-k retrieval accuracy:

In [62]:
def reactant_top_k_accuracy(
    y_true_classes,
    predicted_classes,
    true_reactants,
    k=5
):
    hits = 0
    for cls, true_r in zip(predicted_classes, true_reactants):
        candidates = predict_reactants(cls, k=k)
        if true_r in candidates:
            hits += 1
    return hits / len(true_reactants)

In [63]:
# Predict reaction classes for test set
predicted_classes = np.argmax(probs, axis=1)

true_reactants_test = df.loc[idx_test, "reactants"].values

for k in [1, 3, 5]:
    acc = reactant_top_k_accuracy(
        y_test,
        predicted_classes,
        true_reactants_test,
        k=k
    )
    print(f"Reactant Top-{k} accuracy: {acc:.3f}")

Reactant Top-1 accuracy: 0.000
Reactant Top-3 accuracy: 0.000
Reactant Top-5 accuracy: 0.000


5) Discussion

- Why is reactant prediction harder than reaction-class prediction?
- Why does top‑k accuracy matter more than top‑1?
- Why don't we retrieve our true reactants? How could this be remedied?
